A comprehension is a concise, readable way to build a collection from an iterable in a single expression. Instead of writing a loop, appending to a list, and returning it, you express the whole idea in one line. Python supports four forms: list comprehensions, dictionary comprehensions, set comprehensions, and generator expressions. Used well, they make code shorter and easier to read. Used poorly, they become unreadable puzzles. This notebook covers all four forms, shows where they shine, and explains when a plain loop is the better choice.

## List Comprehensions

A list comprehension builds a new list by applying an expression to every item in an iterable.

```python
[expression for item in iterable]
```

Reading it left to right: *produce this expression, for each item in this iterable*.

Compare the loop form to the comprehension form — they are equivalent, but the comprehension is more direct:

In [ ]:
# Loop form
squares = []
for n in range(1, 8):
    squares.append(n ** 2)
print(squares)

# Comprehension form — same result, one line
squares = [n ** 2 for n in range(1, 8)]
print(squares)   # [1, 4, 9, 16, 25, 36, 49]

In [ ]:
# Any iterable works — strings, lists, ranges, files, …
words = ["hello", "world", "python"]
uppercased = [w.upper() for w in words]
print(uppercased)   # ['HELLO', 'WORLD', 'PYTHON']

# Extract a field from each dict in a list
people = [
    {"name": "Alice", "age": 30},
    {"name": "Bob",   "age": 25},
    {"name": "Carol", "age": 35},
]
names = [p["name"] for p in people]
print(names)   # ['Alice', 'Bob', 'Carol']

### Filtering with `if`

Add an `if` clause to include only items that satisfy a condition:

```python
[expression for item in iterable if condition]
```

In [ ]:
numbers = range(-5, 6)

# Keep only positives
positives = [n for n in numbers if n > 0]
print(positives)   # [1, 2, 3, 4, 5]

# Keep only evens
evens = [n for n in range(20) if n % 2 == 0]
print(evens)

# Transform AND filter in one expression
scores = {"Alice": 88, "Bob": 54, "Carol": 76, "Dave": 45, "Eve": 91}
high_scorers = [f"{name}: {score}" for name, score in scores.items() if score >= 70]
print(high_scorers)

### `if`/`else` in the Expression

When you want to transform every item but differently depending on a condition, put the `if`/`else` in the **expression** — before the `for`. This is a ternary expression applied to each item, not a filter.

```python
[value_if_true if condition else value_if_false  for item in iterable]
```

The position matters: `if` after the `for` **filters** (some items dropped); `if`/`else` before the `for` **transforms** (every item kept, two possible outputs).

In [ ]:
numbers = range(-4, 5)

# Label every number — no items dropped
labels = ["positive" if n > 0 else ("negative" if n < 0 else "zero") for n in numbers]
print(labels)

# Clamp negative values to 0
clamped = [n if n >= 0 else 0 for n in numbers]
print(clamped)   # [0, 0, 0, 0, 0, 1, 2, 3, 4]

# Mark scores as pass/fail
results = [(name, "pass" if score >= 60 else "fail")
           for name, score in scores.items()]
print(results)

### Nested Loops in Comprehensions

Multiple `for` clauses iterate over nested iterables. The order of `for` clauses reads left to right — the same order as equivalent nested loops written out by hand.

In [ ]:
# Cartesian product — all pairs from two lists
colours = ["red", "green", "blue"]
sizes   = ["S", "M", "L"]

variants = [(c, s) for c in colours for s in sizes]
print(variants)
print(f"{len(variants)} variants")   # 9

In [ ]:
# Flatten a nested list
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]

flat = [item for row in matrix for item in row]
print(flat)   # [1, 2, 3, 4, 5, 6, 7, 8, 9]

# Only keep items meeting a condition from the inner loop
odd_from_matrix = [item for row in matrix for item in row if item % 2 != 0]
print(odd_from_matrix)   # [1, 3, 5, 7, 9]

## Dictionary Comprehensions

Dictionary comprehensions build a dict in a single expression. The syntax is identical to list comprehensions, but uses curly braces and a `key: value` expression.

```python
{key_expr: value_expr for item in iterable if condition}
```

In [ ]:
# Word lengths
words = ["apple", "fig", "banana", "kiwi", "elderberry"]
lengths = {w: len(w) for w in words}
print(lengths)

# Invert a mapping
http_codes = {200: "OK", 404: "Not Found", 500: "Server Error"}
by_message = {msg: code for code, msg in http_codes.items()}
print(by_message)

# Build a lookup table from two parallel lists
keys   = ["host", "port", "debug"]
values = ["localhost", 5432, True]
config = {k: v for k, v in zip(keys, values)}
print(config)

In [ ]:
# Filter while building — keep only high scores
scores = {"Alice": 88, "Bob": 54, "Carol": 76, "Dave": 45, "Eve": 91}
top = {name: score for name, score in scores.items() if score >= 80}
print(top)

# Transform values — apply a 10% curve to all scores
curved = {name: min(100, round(score * 1.1)) for name, score in scores.items()}
print(curved)

## Set Comprehensions

Set comprehensions build a set — duplicates are automatically dropped, and the result is unordered. Syntax is like a list comprehension but with curly braces.

```python
{expression for item in iterable if condition}
```

In [ ]:
words = ["apple", "banana", "apricot", "blueberry", "avocado", "banana"]

# Unique first letters
first_letters = {w[0] for w in words}
print(first_letters)   # {'a', 'b'} — duplicates gone, order not guaranteed

# Unique word lengths
unique_lengths = {len(w) for w in words}
print(sorted(unique_lengths))

# Find all unique characters across a list of strings
tokens = ["abc", "bcd", "cde"]
all_chars = {ch for token in tokens for ch in token}
print(sorted(all_chars))   # ['a', 'b', 'c', 'd', 'e']

## Generator Expressions

A generator expression has the same syntax as a list comprehension but uses **parentheses** instead of brackets. Rather than building the entire list in memory upfront, it produces items **one at a time on demand**.

```python
(expression for item in iterable if condition)
```

Use a generator expression when:
- You only need to iterate over the results once.
- The collection could be very large and you do not want it all in memory.
- You are passing the result directly to a function like `sum()`, `min()`, `max()`, or `any()`.

When a generator expression is the only argument to a function, the outer parentheses can be omitted.

In [ ]:
# List comprehension — builds the full list first, then sums
total_list = sum([n ** 2 for n in range(1_000_000)])

# Generator expression — no intermediate list, items computed on demand
total_gen = sum(n ** 2 for n in range(1_000_000))   # parentheses from sum() serve double duty

print(total_list == total_gen)   # True — same answer

In [ ]:
import sys

# Memory comparison
lst = [n ** 2 for n in range(10_000)]
gen = (n ** 2 for n in range(10_000))

print(f"List size:      {sys.getsizeof(lst):>10,} bytes")
print(f"Generator size: {sys.getsizeof(gen):>10,} bytes")  # tiny — just the generator object

In [ ]:
# Generators work well with any(), all(), min(), max()
prices = [12.5, 8.0, 22.3, 5.5, 17.0]

print(any(p > 20 for p in prices))    # True  — stop as soon as one passes
print(all(p > 0 for p in prices))     # True  — stop as soon as one fails
print(min(p * 1.2 for p in prices))   # cheapest after 20% markup

# Generators are lazy — items not produced until consumed
gen = (print(f"producing {n}") or n for n in range(3))
print("Generator created — nothing produced yet")
first = next(gen)   # pull one item
print(f"First item: {first}")

## Walrus Operator in Comprehensions

The walrus operator (`:=`) can assign a value inside a comprehension expression and reuse it — avoiding a redundant call when both the filter condition and the output expression need the same computed value.

In [ ]:
import math

data = [4, -1, 9, 0, 16, -3, 25]

# Without walrus — sqrt computed twice (once to filter, once in expression)
roots_bad = [math.sqrt(n) for n in data if n > 0]

# With walrus — compute sqrt once, reuse in both the filter test and the output
roots_good = [root for n in data if (root := math.sqrt(n)) > 0 and n > 0]

# More natural use: expensive computation done once, result used for both filter and value
words = ["  hello  ", "", "  world  ", "   ", "python"]
cleaned = [s for w in words if (s := w.strip())]   # assign stripped, keep only non-empty
print(cleaned)   # ['hello', 'world', 'python']

## When to Use Comprehensions — and When Not To

Comprehensions are best when:
- The logic fits on one readable line.
- You are building a new collection (not just doing side effects).
- The transformation or filter is simple and clear.

Reach for a plain loop when:
- The logic is complex enough to need multiple statements or intermediate variables.
- You need to `break` or `continue` based on a condition.
- The body has side effects (printing, writing to a file, modifying state).
- Nesting goes deeper than two levels — readability falls off quickly.

In [ ]:
# Fine as a comprehension — simple, one clear idea
emails = ["  Alice@Example.COM  ", "bob@example.com", "  CAROL@WORK.IO"]
normalised = [e.strip().lower() for e in emails]
print(normalised)

# This is getting hard to read — a loop is clearer
data = ["10", "abc", "7", "", "3.5", "99"]

# Comprehension — works but harder to follow
parsed = [int(x) for x in data if x.isdigit()]
print(parsed)

# Loop — same result, easier to extend later
parsed = []
for x in data:
    if x.isdigit():
        parsed.append(int(x))
print(parsed)

In [ ]:
# Never use a comprehension for pure side effects — the list is thrown away
values = [1, 2, 3, 4, 5]

# BAD — builds a list of Nones just to call print
# [print(v) for v in values]

# GOOD — use a loop
for v in values:
    print(v)

## Putting It Together — Real Patterns

A few practical patterns that come up regularly in production code.

In [ ]:
# Build an index: word → list of line numbers it appears on
lines = [
    "the cat sat on the mat",
    "the cat ate the rat",
    "the rat ran away",
]

from collections import defaultdict

index = defaultdict(list)
for lineno, line in enumerate(lines, start=1):
    for word in set(line.split()):   # set() deduplicates words per line
        index[word].append(lineno)

for word in sorted(index):
    print(f"  {word:<8} → lines {index[word]}")

In [ ]:
# Transpose a matrix using a nested comprehension
matrix = [
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9],
]

transposed = [[row[i] for row in matrix] for i in range(len(matrix[0]))]
for row in transposed:
    print(row)

# zip(*matrix) is the idiomatic Python way
transposed_zip = [list(row) for row in zip(*matrix)]
for row in transposed_zip:
    print(row)

In [ ]:
# Pipeline: parse, filter, transform, aggregate — all with generators (no intermediate lists)
raw_data = ["12", "abc", "7", "3", "xyz", "99", "4", "51"]

total = sum(
    n * n
    for x in raw_data
    if x.isdigit()
    for n in [int(x)]    # bind int(x) to n to avoid calling int() twice
    if n % 2 != 0        # keep only odd numbers
)
print(f"Sum of squares of odd numbers: {total}")   # 7²+3²+99²+51² = 49+9+9801+2601 = 12460

## Summary

- **List comprehension**: `[expr for item in iterable if condition]` — builds a list. Filter with `if` after `for` (drops items). Transform conditionally with `if/else` before `for` (keeps all items, two possible outputs).
- **Dictionary comprehension**: `{key: value for item in iterable if condition}` — builds a dict. Natural for inverting mappings, building lookup tables, or filtering/transforming an existing dict.
- **Set comprehension**: `{expr for item in iterable if condition}` — builds a set; duplicates dropped automatically.
- **Generator expression**: `(expr for item in iterable if condition)` — lazy, memory-efficient; items produced on demand. Ideal for large data, one-pass iteration, and feeding `sum()`, `any()`, `all()`, `min()`, `max()`.
- **Nested `for` clauses** read left to right, matching the order of nested loops. Two levels is usually the maximum before readability suffers.
- **Walrus operator** (`:=`) inside a comprehension assigns a computed value once and reuses it in both the filter condition and the output expression.
- Comprehensions are for **building collections** from transformations and filters. Use a plain loop for side effects, complex multi-step logic, or anything that needs `break`/`continue`.